# France–Germany sovereign spreads and implied default probabilities

This Google Colab notebook downloads daily French OAT and German Bund yields from official sources, keeps observations from the beginning of the selected year, and interactively charts:

- France and Germany yields at 2, 5, and 10 years;
- OAT–Bund spreads in basis points;
- cumulative spread-implied default probabilities at 2, 5, and 10 years; and
- conditional and unconditional default probabilities for the 2–5-year and 5–10-year intervals.

The charts use Plotly, so you can hover over any date, zoom into a period, and hide or show individual series from the legend.


## Interpretation and model assumptions

The OAT–Bund spread is used as a market-implied credit-risk proxy. It is not a literal forecast of France's physical default probability. Sovereign spreads also contain liquidity, inflation, fiscal, political, safe-haven, and market-segmentation premia.

For recovery rate R, loss-given-default is LGD = 1 − R. The simplified reduced-form approximation used here is:

- hazard rate = spread / LGD;
- survival probability S(T) = exp(−spread × T / LGD);
- cumulative PD(0,T) = 1 − S(T); and
- conditional PD(a,b) = 1 − S(b) / S(a).

The default probabilities are therefore spread-implied and risk-neutral in spirit. They should be interpreted as relative market indicators, not as standalone actuarial forecasts.


In [ ]:
# Colab normally includes these packages. This check also makes the notebook
# self-contained if it is opened in a fresh Jupyter runtime.
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("plotly") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "plotly"])
import io
from urllib.parse import quote
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")



## Configuration

YEAR controls the beginning of the time series. END_DATE is set automatically to today's date when the notebook runs. Change YEAR if you want to study a different year.

The Deutsche Bundesbank API is public. The notebook first tries the Banque de France public CSV export; if that is unavailable, it falls back to the Webstat API. The API requires a free account/API key. Add the key to WEBSTAT_API_KEY only if the fallback is needed.




In [ ]:
# Study window: from 1 January of YEAR through today's date.
YEAR = 2026
START_DATE = f"{YEAR}-01-01"
END_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")

# Credit-model assumptions. These affect every PD estimate.
RECOVERY_RATE = 0.40
LOSS_GIVEN_DEFAULT = 1.0 - RECOVERY_RATE

# Optional Banque de France Webstat API key.
WEBSTAT_API_KEY = ""

# A negative OAT–Bund spread should not create a negative PD.
# If True, negative spreads are floored at zero only for the PD calculation;
# the raw spread remains visible in the spread charts.
FLOOR_NEGATIVE_SPREADS_AT_ZERO = True

MATURITIES = [2, 5, 10]

print(f"Observation window: {START_DATE} through {END_DATE}")
print(f"Recovery rate: {RECOVERY_RATE:.0%}; LGD: {LOSS_GIVEN_DEFAULT:.0%}")


## Official data sources and series

France uses the Banque de France CNO-TEC daily constant-maturity series. Webstat identifies EURONEXT as the source:

- 2 years: FM.D.FR.EUR.FR2.BB.FRMOYTEC2.HSTA
- 5 years: FM.D.FR.EUR.FR2.BB.FRMOYTEC5.HSTA
- 10 years: FM.D.FR.EUR.FR2.BB.FRMOYTEC10.HSTA

Germany uses the Deutsche Bundesbank daily Svensson term-structure estimates for zero-coupon yields on federal securities:

- 2 years: BBSIS.D.I.ZST.ZI.EUR.S1311.B.A604.R02XX.R.A.A._Z._Z.A
- 5 years: BBSIS.D.I.ZST.ZI.EUR.S1311.B.A604.R05XX.R.A.A._Z._Z.A
- 10 years: BBSIS.D.I.ZST.ZI.EUR.S1311.B.A604.R10XX.R.A.A._Z._Z.A

The notebook compares matched dates only. This avoids calculating a spread from two different holiday calendars or publication dates.


In [ ]:
FRANCE_SERIES = {
    2: "FM.D.FR.EUR.FR2.BB.FRMOYTEC2.HSTA",
    5: "FM.D.FR.EUR.FR2.BB.FRMOYTEC5.HSTA",
    10: "FM.D.FR.EUR.FR2.BB.FRMOYTEC10.HSTA",
}

GERMANY_SERIES = {
    2: "BBSIS.D.I.ZST.ZI.EUR.S1311.B.A604.R02XX.R.A.A._Z._Z.A",
    5: "BBSIS.D.I.ZST.ZI.EUR.S1311.B.A604.R05XX.R.A.A._Z._Z.A",
    10: "BBSIS.D.I.ZST.ZI.EUR.S1311.B.A604.R10XX.R.A.A._Z._Z.A",
}

BDF_API_URL = (
    "https://webstat.banque-france.fr/api/explore/v2.1/"
    "catalog/datasets/observations/exports/json"
)
BDF_CSV_EXPORT_BASE_URL = "https://webstat.banque-france.fr/export/csv/fr/catalog/FM"
BUNDESBANK_API_URL = "https://api.statistiken.bundesbank.de/rest/data/BBSIS"


def standardise_observations(records, date_candidates, value_candidates):
    """Convert an API response into a date/yield_pct table."""
    # Banque de France returns a list for the JSON export. This dictionary
    # handling also tolerates common API wrapper responses.
    if isinstance(records, dict):
        records = records.get("results", records.get("data", records))

    frame = pd.DataFrame(records)
    if frame.empty:
        raise ValueError("The API returned no observations for the requested series.")

    date_col = next((c for c in date_candidates if c in frame.columns), None)
    value_col = next((c for c in value_candidates if c in frame.columns), None)
    if date_col is None or value_col is None:
        raise ValueError(
            f"Could not identify date/value columns. Returned columns: {list(frame.columns)}"
        )

    clean = frame[[date_col, value_col]].copy()
    clean.columns = ["date", "yield_pct"]

    # UTC parsing handles either date-only values or timestamps with a timezone.
    clean["date"] = pd.to_datetime(clean["date"], errors="coerce", utc=True).dt.tz_localize(None)
    clean["yield_pct"] = pd.to_numeric(clean["yield_pct"], errors="coerce")
    clean = clean.dropna().sort_values("date")
    clean = clean[
        (clean["date"] >= pd.Timestamp(START_DATE))
        & (clean["date"] <= pd.Timestamp(END_DATE))
    ]
    return clean.drop_duplicates("date").reset_index(drop=True)


def download_france_series(maturity):
    """Download one official Banque de France CNO-TEC daily series.

    Webstat documents a public long-format CSV export. We try that first,
    because it avoids an API key. If the export is unavailable, we use the
    authenticated JSON API and give a clear message when a key is missing.
    """
    series_key = FRANCE_SERIES[maturity]

    # Public full-history CSV export; filter to the selected year locally.
    csv_url = f"{BDF_CSV_EXPORT_BASE_URL}/{quote(series_key, safe='')}"
    csv_response = requests.get(
        csv_url,
        headers={"Accept": "text/csv"},
        timeout=45,
    )
    if csv_response.ok and "time_period_start" in csv_response.text:
        raw = pd.read_csv(io.StringIO(csv_response.text), sep=";", dtype=str)
        raw.columns = [str(c).strip().lstrip("\ufeff") for c in raw.columns]
        raw["obs_value"] = (
            raw["obs_value"].astype(str).str.replace(",", ".", regex=False)
        )
        clean = standardise_observations(
            raw,
            date_candidates=["time_period_start"],
            value_candidates=["obs_value"],
        )
        clean["maturity_years"] = maturity
        clean["country"] = "France"
        return clean

    # Authenticated JSON API fallback. The official guide says that a missing
    # or unknown key can appear as a 404 NotFoundResource response, which is
    # why the error below explicitly mentions authentication.
    if not WEBSTAT_API_KEY:
        raise RuntimeError(
            "The public Banque de France CSV export was unavailable, and the "
            "Webstat API requires an API key. Create a free Webstat account, "
            "generate an API key, put it in WEBSTAT_API_KEY, and rerun this cell. "
            f"CSV response status was {csv_response.status_code}."
        )

    params = {
        "select": "time_period_start,obs_value",
        "refine": f'series_key:"{series_key}"',
        "where": (
            f"time_period_start >= date'{START_DATE}' "
            f"AND time_period_start <= date'{END_DATE}'"
        ),
        "order_by": "time_period_start",
        "limit": 10000,
    }
    response = requests.get(
        BDF_API_URL,
        params=params,
        headers={
            "Accept": "application/json",
            "Authorization": f"Apikey {WEBSTAT_API_KEY}",
        },
        timeout=45,
    )
    if response.status_code in (401, 403, 404):
        raise RuntimeError(
            "Banque de France Webstat rejected the API request. Check that "
            "WEBSTAT_API_KEY is valid and uses the format Apikey <key>."
        )
    response.raise_for_status()

    clean = standardise_observations(
        response.json(),
        date_candidates=["time_period_start", "TIME_PERIOD", "date"],
        value_candidates=["obs_value", "OBS_VALUE", "value"],
    )
    clean["maturity_years"] = maturity
    clean["country"] = "France"
    return clean

def download_germany_series(maturity):
    """Download one official daily Bundesbank Svensson curve series."""
    series_key = GERMANY_SERIES[maturity]
    # The mapping stores the full SDMX identifier, including the BBSIS flow
    # prefix. The endpoint receives the flow prefix separately.
    key_without_flow = series_key.split("BBSIS.", 1)[-1]
    url = f"{BUNDESBANK_API_URL}/{key_without_flow}"
    last_error = None

    # The Bundesbank documents the exact format name "sdmx_csv". Keep "csv"
    # as a fallback for older endpoint behavior.
    for format_name in ["sdmx_csv", "csv"]:
        try:
            response = requests.get(
                url,
                params={
                    "startPeriod": START_DATE,
                    "endPeriod": END_DATE,
                    "format": format_name,
                    "lang": "en",
                },
                headers={"Accept": "text/csv"},
                timeout=45,
            )
            response.raise_for_status()

            # SDMX-CSV is RFC-4180 comma-separated data.
            raw = pd.read_csv(io.StringIO(response.text), sep=",")
            raw.columns = [str(c).strip().lstrip("\ufeff") for c in raw.columns]

            clean = standardise_observations(
                raw,
                date_candidates=["TIME_PERIOD", "time_period_start", "date"],
                value_candidates=["OBS_VALUE", "obs_value", "value"],
            )
            clean["maturity_years"] = maturity
            clean["country"] = "Germany"
            return clean
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        f"Could not download Bundesbank data for {maturity} years. "
        f"Last error: {last_error}"
    )


# Download all requested maturities for both countries.
france_long = pd.concat(
    [download_france_series(maturity) for maturity in MATURITIES],
    ignore_index=True,
)
germany_long = pd.concat(
    [download_germany_series(maturity) for maturity in MATURITIES],
    ignore_index=True,
)

print(f"France observations: {len(france_long):,}")
print(f"Germany observations: {len(germany_long):,}")
print(f"France dates: {france_long['date'].min().date()} to {france_long['date'].max().date()}")
print(f"Germany dates: {germany_long['date'].min().date()} to {germany_long['date'].max().date()}")





## Align data and calculate daily OAT–Bund spreads

Yields are published in percent per annum. A 100-basis-point spread equals one percentage point, so the difference between two percentage yields is multiplied by 100 to obtain basis points.

The inner join keeps dates for which both official sources have an observation at the same maturity. This is a conservative way to avoid holiday/calendar mismatches.


In [ ]:
def to_wide(frame, prefix):
    # One column per maturity makes the later calculations easy to audit.
    wide = frame.pivot(index="date", columns="maturity_years", values="yield_pct")
    wide = wide.rename(columns={m: f"{prefix}_{m}y_yield_pct" for m in MATURITIES})
    wide.index.name = "date"
    return wide


france_wide = to_wide(france_long, "france")
germany_wide = to_wide(germany_long, "germany")

# Only matched dates are used for the spread and PD calculations.
time_series = france_wide.join(germany_wide, how="inner").sort_index().dropna()
if time_series.empty:
    raise ValueError("No matched France/Germany dates were returned. Check API access and dates.")

for maturity in MATURITIES:
    france_col = f"france_{maturity}y_yield_pct"
    germany_col = f"germany_{maturity}y_yield_pct"
    spread_col = f"spread_{maturity}y_bps"

    # Percent minus percent gives percentage points; times 100 gives bps.
    time_series[spread_col] = (
        time_series[france_col] - time_series[germany_col]
    ) * 100

print(f"Matched observations: {len(time_series):,}")
print(f"Matched dates: {time_series.index.min().date()} to {time_series.index.max().date()}")
display(time_series.tail())


## Calculate cumulative and interval default probabilities for every date

For each date, the observed spread is converted into a survival curve using the recovery-rate assumption. This produces a time series rather than a single snapshot.

The interval measures are:

- conditional PD: probability of default during the interval, conditional on survival to its beginning;
- unconditional contribution: that interval's contribution to total default probability from today to the interval end.

The 2–5 and 5–10 interval hazards are calculated from the change in cumulative hazard across the endpoints.


In [ ]:
for maturity in MATURITIES:
    raw_spread_decimal = time_series[f"spread_{maturity}y_bps"] / 10000.0

    # Keep the raw spread in the data, but prevent a negative spread from
    # becoming a nonsensical negative default probability.
    if FLOOR_NEGATIVE_SPREADS_AT_ZERO:
        spread_decimal = raw_spread_decimal.clip(lower=0.0)
    else:
        spread_decimal = raw_spread_decimal

    time_series[f"hazard_{maturity}y_pct_pa"] = (
        spread_decimal / LOSS_GIVEN_DEFAULT * 100
    )
    time_series[f"survival_{maturity}y_pct"] = (
        np.exp(-spread_decimal * maturity / LOSS_GIVEN_DEFAULT) * 100
    )
    time_series[f"cumulative_pd_{maturity}y_pct"] = (
        100 - time_series[f"survival_{maturity}y_pct"]
    )

# Conditional and unconditional interval probabilities.
for start, end in [(2, 5), (5, 10)]:
    survival_start = time_series[f"survival_{start}y_pct"] / 100
    survival_end = time_series[f"survival_{end}y_pct"] / 100

    time_series[f"pd_{start}to{end}y_conditional_pct"] = (
        1 - survival_end / survival_start
    ) * 100
    time_series[f"pd_{start}to{end}y_unconditional_pct"] = (
        survival_start - survival_end
    ) * 100

latest_date = time_series.index[-1]
latest = time_series.iloc[[-1]].T.rename(columns={latest_date: "Latest value"})
display(latest)


## Latest values and year-to-date changes

This table reports the first matched observation, the latest matched observation, and the change between them. Yields are in percent, spreads are in basis points, and PDs are percentage points.


In [ ]:
summary_columns = []
for maturity in MATURITIES:
    summary_columns += [
        f"france_{maturity}y_yield_pct",
        f"germany_{maturity}y_yield_pct",
        f"spread_{maturity}y_bps",
        f"cumulative_pd_{maturity}y_pct",
    ]
summary_columns += [
    "pd_2to5y_conditional_pct",
    "pd_2to5y_unconditional_pct",
    "pd_5to10y_conditional_pct",
    "pd_5to10y_unconditional_pct",
]

summary = pd.DataFrame(
    {
        "First matched observation": time_series[summary_columns].iloc[0],
        "Latest matched observation": time_series[summary_columns].iloc[-1],
    }
)
summary["Change"] = (
    summary["Latest matched observation"] - summary["First matched observation"]
)
summary.index.name = "Measure"
display(summary.round(4))


## Interactive Plotly chart: yields

Hover over the lines to read the date and yield. The legend is interactive: click a series to hide it, or double-click one series to isolate it. Use the Plotly toolbar to zoom, pan, or reset the view.


In [ ]:
fig_yields = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=("2-year yields", "5-year yields", "10-year yields"),
)

for row, maturity in enumerate(MATURITIES, start=1):
    fig_yields.add_trace(
        go.Scatter(
            x=time_series.index,
            y=time_series[f"france_{maturity}y_yield_pct"],
            mode="lines",
            name=f"France OAT {maturity}Y",
            hovertemplate="%{x|%Y-%m-%d}<br>France: %{y:.3f}%<extra></extra>",
        ),
        row=row,
        col=1,
    )
    fig_yields.add_trace(
        go.Scatter(
            x=time_series.index,
            y=time_series[f"germany_{maturity}y_yield_pct"],
            mode="lines",
            name=f"Germany Bund {maturity}Y",
            hovertemplate="%{x|%Y-%m-%d}<br>Germany: %{y:.3f}%<extra></extra>",
        ),
        row=row,
        col=1,
    )
    fig_yields.update_yaxes(title_text="Yield (%)", row=row, col=1)

fig_yields.update_layout(
    title=f"Daily France and Germany yields since {START_DATE}",
    height=900,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig_yields.update_xaxes(title_text="Date", row=3, col=1)
fig_yields.show()


## Interactive Plotly chart: OAT–Bund spreads

This chart shows whether the France–Germany yield differential is widening or narrowing at each maturity. The 10-year line is the direct time-series version of the spread discussed in the original question.


In [ ]:
fig_spreads = go.Figure()

for maturity in MATURITIES:
    fig_spreads.add_trace(
        go.Scatter(
            x=time_series.index,
            y=time_series[f"spread_{maturity}y_bps"],
            mode="lines",
            name=f"{maturity}Y spread",
            hovertemplate="%{x|%Y-%m-%d}<br>%{y:.1f} bps<extra></extra>",
        )
    )

fig_spreads.add_hline(y=0, line_dash="dot", line_color="gray")
fig_spreads.update_layout(
    title=f"France–Germany OAT–Bund spreads since {START_DATE}",
    xaxis_title="Date",
    yaxis_title="Spread (basis points)",
    height=550,
    template="plotly_white",
    hovermode="x unified",
)
fig_spreads.show()


## Interactive Plotly chart: cumulative spread-implied default probabilities

These curves show the estimated probability of at least one default by each horizon, using the daily observed spreads and the chosen recovery assumption. They are model outputs, not observed default frequencies.


In [ ]:
fig_cumulative_pd = go.Figure()

for maturity in MATURITIES:
    fig_cumulative_pd.add_trace(
        go.Scatter(
            x=time_series.index,
            y=time_series[f"cumulative_pd_{maturity}y_pct"],
            mode="lines",
            name=f"{maturity}Y cumulative PD",
            hovertemplate="%{x|%Y-%m-%d}<br>%{y:.2f}%<extra></extra>",
        )
    )

fig_cumulative_pd.update_layout(
    title=(
        f"Cumulative spread-implied default probability since {START_DATE}"
        f" (recovery assumption: {RECOVERY_RATE:.0%})"
    ),
    xaxis_title="Date",
    yaxis_title="Cumulative PD (%)",
    height=550,
    template="plotly_white",
    hovermode="x unified",
)
fig_cumulative_pd.show()


## Interactive Plotly chart: interval default probabilities

The first panel is conditional on survival to the beginning of the interval. For example, the 5–10-year line is the probability of default between years 5 and 10 given survival through year 5.

The second panel shows the same intervals as unconditional contributions to total default probability.


In [ ]:
fig_intervals = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=("Conditional interval PD", "Unconditional contribution to total PD"),
)

for start, end in [(2, 5), (5, 10)]:
    fig_intervals.add_trace(
        go.Scatter(
            x=time_series.index,
            y=time_series[f"pd_{start}to{end}y_conditional_pct"],
            mode="lines",
            name=f"{start}–{end}Y conditional PD",
            legendgroup=f"{start}-{end}",
            hovertemplate="%{x|%Y-%m-%d}<br>%{y:.2f}%<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig_intervals.add_trace(
        go.Scatter(
            x=time_series.index,
            y=time_series[f"pd_{start}to{end}y_unconditional_pct"],
            mode="lines",
            name=f"{start}–{end}Y unconditional contribution",
            legendgroup=f"{start}-{end}",
            showlegend=False,
            hovertemplate="%{x|%Y-%m-%d}<br>%{y:.2f}%<extra></extra>",
        ),
        row=2,
        col=1,
    )

fig_intervals.update_yaxes(title_text="Conditional PD (%)", rangemode="tozero", row=1, col=1)
fig_intervals.update_yaxes(title_text="PD contribution (%)", rangemode="tozero", row=2, col=1)
fig_intervals.update_xaxes(title_text="Date", row=2, col=1)
fig_intervals.update_layout(
    title=f"Interval default probabilities since {START_DATE}",
    height=750,
    template="plotly_white",
    hovermode="x unified",
)
fig_intervals.show()


## Optional: recovery-rate sensitivity at the latest date

The same spread implies a larger PD when recovery is assumed to be lower, because loss-given-default is higher. This table shows the sensitivity at the latest matched observation.


In [ ]:
latest_row = time_series.iloc[-1]

def latest_pd_for_recovery(recovery_rate):
    lgd = 1.0 - recovery_rate
    values = {}
    for maturity in MATURITIES:
        spread_decimal = latest_row[f"spread_{maturity}y_bps"] / 10000.0
        if FLOOR_NEGATIVE_SPREADS_AT_ZERO:
            spread_decimal = max(spread_decimal, 0.0)
        values[f"{maturity}Y cumulative PD (%)"] = (
            1 - np.exp(-spread_decimal * maturity / lgd)
        ) * 100
    return values

sensitivity = pd.DataFrame(
    [
        {"Recovery rate": f"{r:.0%}", **latest_pd_for_recovery(r)}
        for r in [0.00, 0.40, 0.60]
    ]
).set_index("Recovery rate")

print(f"Latest matched observation: {latest_date.date()}")
display(sensitivity.round(4))


## Caveats and source links

1. The spread is not a pure default-risk premium. Liquidity, market positioning, inflation risk, fiscal expectations, politics, and Germany's safe-haven status can all move it.
2. The calculation is a simple constant-hazard approximation. A proper term structure would use a bootstrapped hazard curve, discounting, recovery conventions, and possibly CDS rather than a bond spread.
3. Bond yields and source conventions may differ slightly. The notebook uses official constant-maturity/term-structure series and compares matched dates, but it does not force coupon, duration, or liquidity matching.
4. The results are best interpreted as relative, market-implied indicators. Do not treat them as a standalone estimate of France's actuarial or physical probability of default.

Official sources:

- Banque de France daily CNO-TEC market-interest page: https://www.banque-france.fr/fr/statistiques/taux-et-cours/indices-obligataires-2026-10-02
- Banque de France Webstat API guide: https://webstat.banque-france.fr/pages/guide-migration-api/
- Banque de France 2-year series: https://webstat.banque-france.fr/fr/catalogue/fm/FM.D.FR.EUR.FR2.BB.FRMOYTEC2.HSTA
- Banque de France 5-year series: https://webstat.banque-france.fr/fr/catalogue/fm/FM.D.FR.EUR.FR2.BB.FRMOYTEC5.HSTA
- Banque de France 10-year series: https://webstat.banque-france.fr/fr/catalogue/fm/FM.D.FR.EUR.FR2.BB.FRMOYTEC10.HSTA
- Deutsche Bundesbank daily yield curve: https://www.bundesbank.de/de/statistiken/geld-und-kapitalmaerkte/zinssaetze-und-renditen/taegliche-zinsstruktur-fuer-boersennotierte-bundeswertpapiere-650724
- Deutsche Bundesbank SDMX API documentation: https://www.bundesbank.de/en/statistics/time-series-databases/help-for-sdmx-web-service/web-service-interface-data
- IMF discussion of spread-based default-probability approximations: https://www.elibrary.imf.org/view/journals/001/2006/104/article-A001-en.xml


## CDS comparison: public 5-year sovereign CDS quotes

This section adds a separately sourced 5-year sovereign CDS comparison. The CDS spread is converted to a simple cumulative five-year risk-neutral PD using the same recovery-rate assumption as the bond-spread calculation.

The notebook uses the public Boursorama quote pages because their HTML chart data can be retrieved without an API key. The France and Germany pages identify the instruments as 5-year sovereign CDS and expose a daily chart. The code checks the response and parses the chart before doing any calculation.

The public Investing.com series is useful as a cross-check, but its France and Germany instruments are explicitly labelled USD: France `FRGV5YUSAC=R` and Germany `DEGV5YUSAC=R`. The Boursorama pages used below do not state the quote currency, so the notebook does not silently label those observations as EUR or USD. This distinction matters because the CDS premium currency can create a currency basis.


In [ ]:
# Public CDS pages and contract metadata.
# A CDS is normally quoted as an annual spread in basis points (bps),
# not as a bond price. The quote currency is kept explicit because the
# public Investing comparison series is labelled USD, while Boursorama
# does not state the currency on these quote pages.
import html
import json
import re

CDS_TENOR_YEARS = 5
CDS_TICKERS = {
    "France": {
        "symbol": "3xFRTR",
        "reference_entity": "French Republic",
        "url": "https://www.boursorama.com/cours/cds/3xFRTR/",
    },
    "Germany": {
        "symbol": "3xDBR",
        "reference_entity": "Federal Republic of Germany",
        "url": "https://www.boursorama.com/cours/cds/3xDBR/",
    },
}

INVESTING_CDS_METADATA = {
    "France": {
        "symbol": "FRGV5YUSAC=R",
        "quote_currency": "USD",
        "url": "https://fr.investing.com/rates-bonds/france-cds-5-years-usd-historical-data",
    },
    "Germany": {
        "symbol": "DEGV5YUSAC=R",
        "quote_currency": "USD",
        "url": "https://www.investing.com/rates-bonds/germany-cds-5-year-usd-historical-data",
    },
}

def _period_tokens_to_dates(period_tokens, anchor_date):
    """Reconstruct dates from Boursorama's day/month chart labels."""
    if not period_tokens:
        return pd.DatetimeIndex([])

    month_days = [(int(token[3:5]), int(token[:2])) for token in period_tokens]
    # The chart usually spans one year and omits the year from each label.
    # If the first month/day is later than the last, the chart crossed New Year.
    year = anchor_date.year - int(month_days[0] > month_days[-1])
    dates = []
    previous_month_day = None
    for month, day in month_days:
        month_day = (month, day)
        if previous_month_day is not None and month_day < previous_month_day:
            year += 1
        dates.append(pd.Timestamp(year=year, month=month, day=day))
        previous_month_day = month_day
    return pd.DatetimeIndex(dates)

def download_boursorama_cds_history(country):
    """Download and validate the public one-year 5Y CDS chart for a country."""
    metadata = CDS_TICKERS[country]
    response = requests.get(
        metadata["url"],
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=45,
    )
    response.raise_for_status()
    page = response.text

    # Select the quote faceplate for this CDS rather than another market
    # instrument that may also appear in the page header.
    faceplate = re.search(
        rf'data-faceplate-symbol="{re.escape(metadata["symbol"])}".*?'
        r'data-ist-init="([^"]+)"',
        page,
        flags=re.DOTALL,
    )
    if faceplate is None:
        raise ValueError(f"Could not find the {country} CDS quote metadata.")
    quote_metadata = json.loads(html.unescape(faceplate.group(1)))

    # Boursorama embeds the chart as JSON.parse('...') in the page.
    chart_match = re.search(
        r"var chart\s*=\s*JSON\.parse\('(.*?)'\);",
        page,
        flags=re.DOTALL,
    )
    if chart_match is None:
        raise ValueError(f"Could not find the {country} CDS chart data.")
    chart = json.loads(chart_match.group(1))

    series = None
    for candidate in chart.get("data", {}).get("amChartData", {}).values():
        if isinstance(candidate, list) and candidate and {"period", "value"}.issubset(candidate[0]):
            series = candidate
            break
    if series is None:
        raise ValueError(f"Could not identify the {country} CDS time series.")

    raw = pd.DataFrame(series)
    anchor_date = pd.Timestamp(quote_metadata["tradeDate"])
    raw["date"] = _period_tokens_to_dates(raw["period"].tolist(), anchor_date)
    raw["cds_5y_bps"] = pd.to_numeric(raw["value"], errors="coerce")
    raw = raw.dropna(subset=["date", "cds_5y_bps"]).sort_values("date")
    raw = raw[(raw["date"] >= pd.Timestamp(START_DATE)) & (raw["date"] <= pd.Timestamp(END_DATE))]
    if raw.empty:
        raise ValueError(f"The {country} CDS page returned no observations in the selected window.")

    raw["country"] = country
    raw["reference_entity"] = metadata["reference_entity"]
    raw["source"] = "Boursorama"
    raw["source_url"] = metadata["url"]
    raw["quote_currency"] = "Not stated on Boursorama page"
    raw["tenor_years"] = CDS_TENOR_YEARS
    raw["quote_timestamp"] = anchor_date
    return raw[[
        "date", "country", "reference_entity", "cds_5y_bps",
        "quote_currency", "tenor_years", "source", "source_url",
        "quote_timestamp",
    ]].drop_duplicates("date").reset_index(drop=True)

cds_long = pd.concat(
    [download_boursorama_cds_history(country) for country in CDS_TICKERS],
    ignore_index=True,
)
if cds_long.empty:
    raise ValueError("No public CDS observations were downloaded.")

cds_wide = cds_long.pivot(index="date", columns="country", values="cds_5y_bps")
cds_wide = cds_wide.rename(columns={
    "France": "cds_france_5y_bps",
    "Germany": "cds_germany_5y_bps",
})

# Restrict the comparison to dates where both the bond-spread data and
# both CDS quotes are available. This avoids carrying stale CDS values
# across holidays or missing observations.
cds_comparison = time_series.join(cds_wide, how="inner").sort_index().dropna()
if cds_comparison.empty:
    raise ValueError("No dates match the bond and CDS datasets. Check the selected YEAR.")

for country in ["France", "Germany"]:
    cds_decimal = cds_comparison[f"cds_{country.lower()}_5y_bps"] / 10000.0
    if FLOOR_NEGATIVE_SPREADS_AT_ZERO:
        cds_decimal = cds_decimal.clip(lower=0.0)
    cds_comparison[f"cds_{country.lower()}_5y_hazard_pct_pa"] = (
        cds_decimal / LOSS_GIVEN_DEFAULT * 100
    )
    cds_comparison[f"cds_{country.lower()}_5y_cumulative_pd_pct"] = (
        1.0 - np.exp(-cds_decimal * CDS_TENOR_YEARS / LOSS_GIVEN_DEFAULT)
    ) * 100

cds_comparison["cds_france_minus_germany_5y_bps"] = (
    cds_comparison["cds_france_5y_bps"] - cds_comparison["cds_germany_5y_bps"]
)

print("CDS source: Boursorama public 5Y quote pages")
print("Boursorama quote currency: not stated; Investing comparison series: USD-labelled")
print(f"CDS matched observations: {len(cds_comparison):,}")
print(f"CDS matched dates: {cds_comparison.index.min().date()} to {cds_comparison.index.max().date()}")
display(cds_comparison[[
    "cds_france_5y_bps", "cds_germany_5y_bps",
    "cds_france_minus_germany_5y_bps",
    "cds_france_5y_cumulative_pd_pct",
    "cds_germany_5y_cumulative_pd_pct",
]].tail().round(4))


## Plotly chart: bond spreads versus CDS spreads

The upper panel retains the OAT–Bund yield spreads. The lower panel shows the absolute France and Germany 5-year CDS spreads and their difference. The bond spread and the CDS difference are related, but they are not expected to match one-for-one: the bond spread includes liquidity, duration, safe-haven and rate effects, while CDS has its own currency, basis, liquidity and contract effects.


In [ ]:
fig_credit_spreads = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=("France–Germany bond-yield spreads", "5-year sovereign CDS spreads"),
)

for maturity in MATURITIES:
    fig_credit_spreads.add_trace(
        go.Scatter(
            x=cds_comparison.index,
            y=cds_comparison[f"spread_{maturity}y_bps"],
            mode="lines",
            name=f"OAT–Bund {maturity}Y",
            legendgroup="bond",
            hovertemplate="%{x|%Y-%m-%d}<br>OAT–Bund: %{y:.1f} bps<extra></extra>",
        ),
        row=1,
        col=1,
    )

for column, label, color in [
    ("cds_france_5y_bps", "France 5Y CDS", "#d62728"),
    ("cds_germany_5y_bps", "Germany 5Y CDS", "#1f77b4"),
    ("cds_france_minus_germany_5y_bps", "France–Germany CDS difference", "#9467bd"),
]:
    fig_credit_spreads.add_trace(
        go.Scatter(
            x=cds_comparison.index,
            y=cds_comparison[column],
            mode="lines",
            name=label,
            line=dict(color=color),
            legendgroup="cds",
            hovertemplate="%{x|%Y-%m-%d}<br>CDS: %{y:.1f} bps<extra></extra>",
        ),
        row=2,
        col=1,
    )

fig_credit_spreads.add_hline(y=0, line_dash="dot", line_color="gray", row=1, col=1)
fig_credit_spreads.add_hline(y=0, line_dash="dot", line_color="gray", row=2, col=1)
fig_credit_spreads.update_yaxes(title_text="Basis points", row=1, col=1)
fig_credit_spreads.update_yaxes(title_text="Basis points", row=2, col=1)
fig_credit_spreads.update_xaxes(title_text="Date", row=2, col=1)
fig_credit_spreads.update_layout(
    title=f"Bond and CDS credit-risk measures since {START_DATE}",
    height=800,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig_credit_spreads.show()


## Plotly chart: CDS-implied PD versus the OAT–Bund PD proxy

The France CDS curve is an absolute France credit-risk measure. The OAT–Bund curve is a relative France-versus-Germany bond-spread proxy. The German CDS curve is shown separately as a reminder that Germany is not a zero-risk entity.

These curves use the same flat-hazard approximation and recovery assumption. They are not equivalent measures and should not be added together.


In [ ]:
fig_cds_pd = go.Figure()

fig_cds_pd.add_trace(
    go.Scatter(
        x=cds_comparison.index,
        y=cds_comparison["cumulative_pd_5y_pct"],
        mode="lines",
        name="OAT–Bund 5Y spread PD proxy",
        line=dict(color="#ff7f0e", dash="dash"),
        hovertemplate="%{x|%Y-%m-%d}<br>OAT–Bund proxy: %{y:.2f}%<extra></extra>",
    )
)

for country, color in [("France", "#d62728"), ("Germany", "#1f77b4")]:
    fig_cds_pd.add_trace(
        go.Scatter(
            x=cds_comparison.index,
            y=cds_comparison[f"cds_{country.lower()}_5y_cumulative_pd_pct"],
            mode="lines",
            name=f"{country} 5Y CDS PD",
            line=dict(color=color),
            hovertemplate="%{x|%Y-%m-%d}<br>CDS-implied PD: %{y:.2f}%<extra></extra>",
        )
    )

fig_cds_pd.update_layout(
    title=(
        f"Five-year default-probability comparison since {START_DATE}"
        f" (recovery assumption: {RECOVERY_RATE:.0%})"
    ),
    xaxis_title="Date",
    yaxis_title="Cumulative risk-neutral/model-implied PD (%)",
    template="plotly_white",
    height=550,
    hovermode="x unified",
)
fig_cds_pd.show()


## CDS contract notes: currency, reference bonds and Frexit limitations

### Is the quote in USD?

The public Investing comparison instruments are explicitly labelled USD. That means the premium and settlement cashflows are denominated in USD; it does **not** mean that France's underlying government bonds are USD bonds. A sovereign CDS can be quoted in a foreign currency while referencing a sovereign whose debt is primarily issued in euros. The USD and EUR versions can differ because of funding, collateral and FX-basis effects.

The Boursorama pages used for the time series do not state the quote currency. The notebook therefore records `Not stated on Boursorama page` and does not mix the Boursorama observations with the USD-labelled Investing observations.

### What basket of bonds is used?

This is not an average of the 2-, 5- and 10-year OATs, and it is not a fixed basket. A single-name sovereign CDS references the French Republic or the Federal Republic of Germany as the reference entity. The contract specifies a reference obligation and eligibility rules for obligations that can be delivered or used in settlement after a credit event. The public quote pages do not disclose the exact reference ISIN or a complete deliverable-obligation list.

Under the ISDA definitions, the standard-reference-obligation and deliverable-obligation framework matters for settlement. A credit-event auction or cash settlement can therefore produce a payoff that depends on the eligible bond set and recovery price, rather than simply on the price of one selected OAT.

### Frexit and other CDS loopholes

A Frexit, government collapse, rating downgrade or widening bond spread is not automatically a CDS credit event. Payment depends on the exact confirmation, governing law, applicable ISDA definitions, the selected credit events and the ISDA Determinations Committee process.

A euro exit could become relevant if it caused a contractual failure to pay, a qualifying restructuring, repudiation/moratorium, or a change in the currency of payment that falls within the contract's impermissible-currency rules. But an orderly political exit that preserves the payment terms of the covered obligations would not necessarily trigger a CDS payout. The 2014 ISDA definitions specifically addressed currency-redenomination issues and introduced sovereign asset-package delivery provisions.

The practical limitations are therefore:

- CDS protects against defined contractual credit events, not every form of political or market loss.
- A rating downgrade alone does not trigger payment.
- A bond can lose value without a CDS credit event.
- A restructuring may trigger CDS even when the government continues making some payments.
- The final legal interpretation and settlement mechanism can matter as much as the headline CDS spread.

Sources: [ISDA 2014 Definitions announcement](https://www.isda.org/2014/02/21/isda-publishes-isda-2014-credit-derivatives-definitions/), [ISDA sovereign CDS Q&A](https://www.isda.org/2011/07/27/cds-on-us-sovereign-debt-qa/?subcategories=0), [Investing France 5Y USD CDS](https://fr.investing.com/rates-bonds/france-cds-5-years-usd-historical-data), [Investing Germany 5Y USD CDS](https://www.investing.com/rates-bonds/germany-cds-5-year-usd-historical-data).

The calculations in this notebook are analytical approximations for research. They are not a legal opinion on whether a future Frexit or redenomination would trigger a particular CDS confirmation.
